In [ ]:
# !pip install -q google-adk[a2a]

Agent2Agent (A2A) Communication with ADK

Build multi-agent systems where different agents can communicate and collaborate using the Agent2Agent (A2A) Protocol. You'll learn how to integrate with external agent services and consume remote agents as if they were local.

In this notebook, you'll:

Understand the A2A protocol and when to use it vs sub-agents
Learn common A2A architecture patterns (cross-framework, cross-language, cross-organization)
Expose an ADK agent via A2A using to_a2a()
Consume remote agents using RemoteA2aAgent
Build a product catalog integration system

We'll build a practical e-commerce integration:
1. **Product Catalog Agent** (exposed via A2A) - External vendor service that provides product information
2. **Customer Support Agent** (consumer) - Your internal agent that helps customers by querying product data

```text
┌──────────────────────┐           ┌──────────────────────┐
│ Customer Support     │  ─A2A──▶  │ Product Catalog      │
│ Agent (Consumer)     │           │ Agent (Vendor)       │
│ Your Company         │           │ External Service     │
│ (localhost:8000)     │           │ (localhost:8001)     │
└──────────────────────┘           └──────────────────────┘
```

**Why this justifies A2A:**
- Product Catalog is maintained by an external vendor (you can't modify their code)
- Different organizations with separate systems
- Formal contract needed between services
- Product Catalog could be in a different language/framework


In [ ]:
#IMPORT ADK COMPONENTS

In [ ]:
import json
import requests
import subprocess
import time
import uuid

from google.adk.agents import LlmAgent
from google.adk.agents.remote_a2a_agent import (
    RemoteA2aAgent,
    AGENT_CARD_WELL_KNOWN_PATH,
)

from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Hide additional warnings in the notebook
import warnings

warnings.filterwarnings("ignore")

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# have .env in same root folder with variable:
# GOOGLE_GENAI_USE_VERTEXAI=FALSE
GOOGLE_API_KEY = ""
# Access the API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")

print("GOOGLE_API_KEY loaded successfully")


GOOGLE_API_KEY loaded successfully


In [ ]:
# Simulated product information utility
# In a production environment, this would interface with a live inventory system
def fetch_product_details(item_name: str) -> str:
    """
    Retrieve details for a specified product.

    Args:
        item_name (str): Product identifier (e.g., "iPhone 15 Pro", "MacBook Pro")

    Returns:
        str: Human-readable product details
    """

    # Static inventory snapshot (mock data)
    inventory_data = {
        "iphone 15 pro": "iPhone 15 Pro, $999, Low Stock (8 units), 128GB, Titanium finish",
        "samsung galaxy s24": "Samsung Galaxy S24, $799, In Stock (31 units), 256GB, Phantom Black",
        "dell xps 15": 'Dell XPS 15, $1,299, In Stock (45 units), 15.6" display, 16GB RAM, 512GB SSD',
        "macbook pro 14": 'MacBook Pro 14", $1,999, In Stock (22 units), M3 Pro chip, 18GB RAM, 512GB SSD',
        "sony wh-1000xm5": "Sony WH-1000XM5 Headphones, $399, In Stock (67 units), Noise-canceling, 30hr battery",
        "ipad air": 'iPad Air, $599, In Stock (28 units), 10.9" display, 64GB',
        "lg ultrawide 34": 'LG UltraWide 34" Monitor, $499, Out of Stock, Expected: Next week',
    }

    normalized_name = item_name.strip().lower()

    product_info = inventory_data.get(normalized_name)

    if product_info:
        return f"Product Details: {product_info}"

    product_list = ", ".join(name.title() for name in inventory_data)
    return (
        f"Information for '{item_name}' is unavailable. "
        f"Products currently supported: {product_list}"
    )


In [ ]:
# Initialize an agent responsible for vendor product details
# This agent acts as a bridge to the external product inventory
catalog_lookup_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite"),
    name="catalog_lookup_agent",
    description=(
        "An external vendor-facing agent responsible for sharing "
        "product specifications, pricing, and stock status."
    ),
    instruction="""
    You act as a product information expert representing an external vendor.
    Whenever product-related questions arise, rely on the fetch_product_details tool
    to retrieve accurate catalog data.
    Ensure responses clearly mention pricing, availability, and key specifications.
    If multiple products are referenced, handle each request individually.
    Maintain a professional and customer-friendly tone at all times.
    """,
    tools=[fetch_product_details],  # Attach the catalog query tool
)

print("Catalog Lookup Agent initialized successfully.")


Catalog Lookup Agent initialized successfully.


In [ ]:
# Wrap the catalog agent as an A2A-enabled service
# This transformation:
#   - Exposes the agent via standard A2A endpoints
#   - Generates an agent card automatically
#   - Manages request/response handling under the A2A protocol
catalog_agent_service = to_a2a(
    catalog_lookup_agent,  # Agent instance to expose
    port=8001              # Network port for the A2A server
)

print("✅ Catalog Lookup Agent successfully exposed via A2A!")
print("   Server is configured and ready to launch.")


✅ Catalog Lookup Agent successfully exposed via A2A!
   Server is configured and ready to launch.


In [ ]:
# Persist the catalog agent code into a Python module for uvicorn consumption
catalog_agent_source = '''
import os
from google.adk.agents import LlmAgent
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.models.google_llm import Gemini
from google.genai import types

# Retry policy for transient HTTP failures
http_retry_policy = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

def fetch_product_details(item_name: str) -> str:
    """Return catalog details for a requested product."""
    inventory_snapshot = {
        "iphone 15 pro": "iPhone 15 Pro, $999, Low Stock (8 units), 128GB, Titanium finish",
        "samsung galaxy s24": "Samsung Galaxy S24, $799, In Stock (31 units), 256GB, Phantom Black",
        "dell xps 15": "Dell XPS 15, $1,299, In Stock (45 units), 15.6\\" display, 16GB RAM, 512GB SSD",
        "macbook pro 14": "MacBook Pro 14\\", $1,999, In Stock (22 units), M3 Pro chip, 18GB RAM, 512GB SSD",
        "sony wh-1000xm5": "Sony WH-1000XM5 Headphones, $399, In Stock (67 units), Noise-canceling, 30hr battery",
        "ipad air": "iPad Air, $599, In Stock (28 units), 10.9\\" display, 64GB",
        "lg ultrawide 34": "LG UltraWide 34\\" Monitor, $499, Out of Stock, Expected: Next week",
    }

    key = item_name.strip().lower()

    if key in inventory_snapshot:
        return f"Product Details: {inventory_snapshot[key]}"

    supported_items = ", ".join(name.title() for name in inventory_snapshot.keys())
    return (
        f"Details for '{item_name}' are unavailable. "
        f"Supported products include: {supported_items}"
    )

catalog_agent = LlmAgent(
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=http_retry_policy
    ),
    name="catalog_lookup_agent",
    description=(
        "An external-facing agent that supplies pricing, "
        "availability, and specifications from a vendor catalog."
    ),
    instruction="""
    You represent an external vendor's product information desk.
    For any product-related inquiry, use the fetch_product_details tool
    to retrieve catalog data.
    Ensure responses clearly include pricing, stock status, and specifications.
    When multiple products are mentioned, address each independently.
    Maintain a professional and helpful tone.
    """,
    tools=[fetch_product_details],
)

# Expose the agent using the A2A protocol
app = to_a2a(catalog_agent, port=8001)
'''

In [ ]:
# Write the agent module to disk
agent_file_path = "catalog_agent_server.py"
with open(agent_file_path, "w") as file_handle:
    file_handle.write(catalog_agent_source)

print(f"Catalog agent module written to {agent_file_path}")

Catalog agent module written to catalog_agent_server.py


In [ ]:
# Launch the uvicorn server as a background process
server_handle = subprocess.Popen(
    [
        "uvicorn",
        "catalog_agent_server:app",
        "--host",
        "localhost",
        "--port",
        "8001",
    ],
    cwd=os.getcwd(),  # Use the current working directory
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ},
)

print(" Launching Catalog Lookup Agent server...")
print("   Awaiting readiness confirmation...")

 Launching Catalog Lookup Agent server...
   Awaiting readiness confirmation...


In [ ]:
# Poll the agent card endpoint until the service is ready
max_checks = 30
for check in range(max_checks):
    try:
        result = requests.get(
            "http://localhost:8001/.well-known/agent-card.json",
            timeout=1
        )
        if result.status_code == 200:
            print("\n Catalog Lookup Agent server is live!")
            print("   Base URL : http://localhost:8001")
            print("   Agent Card: http://localhost:8001/.well-known/agent-card.json")
            break
    except requests.exceptions.RequestException:
        time.sleep(5)
        print(".", end="", flush=True)
else:
    print("\n  Server startup not confirmed. Manual verification recommended.")

# Retain the process reference for later shutdown
globals()["catalog_agent_server_process"] = server_handle


.
 Catalog Lookup Agent server is live!
   Base URL : http://localhost:8001
   Agent Card: http://localhost:8001/.well-known/agent-card.json


In [ ]:
# Retrieve the agent metadata from the active A2A service
try:
    card_response = requests.get(
        "http://localhost:8001/.well-known/agent-card.json",
        timeout=5
    )

    if card_response.ok:
        catalog_agent_card = card_response.json()

        print("📋 Catalog Lookup Agent – Agent Card")
        print(json.dumps(catalog_agent_card, indent=2))

        print("\n✨ Agent Overview:")
        print(f"   Agent Name : {catalog_agent_card.get('name')}")
        print(f"   Summary    : {catalog_agent_card.get('description')}")
        print(f"   Endpoint   : {catalog_agent_card.get('url')}")
        print(
            f"   Capabilities: "
            f"{len(catalog_agent_card.get('skills', []))} available"
        )
    else:
        print(
            f" Unable to retrieve agent card "
            f"(HTTP {card_response.status_code})"
        )

except requests.exceptions.RequestException as err:
    print(f" Connection error while fetching agent card: {err}")
    print(
        "   Verify that the Catalog Lookup Agent server "
        "is running before retrying."
    )


📋 Catalog Lookup Agent – Agent Card
{
  "capabilities": {},
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "description": "An external-facing agent that supplies pricing, availability, and specifications from a vendor catalog.",
  "name": "catalog_lookup_agent",
  "preferredTransport": "JSONRPC",
  "protocolVersion": "0.3.0",
  "skills": [
    {
      "description": "An external-facing agent that supplies pricing, availability, and specifications from a vendor catalog. \n    I represent an external vendor's product information desk.\n    For any product-related inquiry, use the fetch_product_details tool\n    to retrieve catalog data.\n    Ensure responses clearly include pricing, stock status, and specifications.\n    When multiple products are mentioned, address each independently.\n    Maintain a professional and helpful tone.\n    ",
      "examples": [],
      "id": "catalog_lookup_agent",
      "name": "model",
      "tags": [
   

In [ ]:
# Instantiate a RemoteA2aAgent as a client-side interface
# This allows other agents (e.g., Customer Support) to interact with the catalog agent transparently
catalog_agent_proxy = RemoteA2aAgent(
    name="catalog_lookup_agent",
    description=(
        "Client proxy for an external vendor’s product catalog agent, "
        "exposing product details and availability over A2A."
    ),
    # Reference the published agent card endpoint for A2A metadata
    agent_card=f"http://localhost:8001{AGENT_CARD_WELL_KNOWN_PATH}",
)

print(" Catalog Lookup Agent proxy initialized successfully!")
print("   Connection target : http://localhost:8001")
print(f"   Agent metadata URL: http://localhost:8001{AGENT_CARD_WELL_KNOWN_PATH}")
print("   Ready for use as a local sub-agent within the support workflow.")


 Catalog Lookup Agent proxy initialized successfully!
   Connection target : http://localhost:8001
   Agent metadata URL: http://localhost:8001/.well-known/agent-card.json
   Ready for use as a local sub-agent within the support workflow.


In [ ]:
# Initialize the Customer Support Agent with access to the remote catalog agent
support_agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite"),
    name="customer_support_agent",
    description=(
        "Customer support assistant designed to assist users with product inquiries, "
        "leveraging a remote catalog agent for accurate information."
    ),
    instruction="""
    You are a professional and approachable customer support agent.

    For any product-related questions:
    1. Query the catalog_lookup_agent sub-agent to fetch product details.
    2. Provide concise and accurate information on price, stock, and specifications.
    3. Notify users of expected restock dates if a product is unavailable.
    4. Maintain a helpful and professional tone throughout.

    Always retrieve product data from the catalog_lookup_agent before responding.
    """,
    sub_agents=[catalog_agent_proxy],  # Register the remote product catalog agent
)

print(" Customer Support Agent initialized successfully!")
print("   Model in use : gemini-2.5-flash-lite")
print(f"   Sub-agents   : {len(support_agent.sub_agents)} (remote catalog agent via A2A)")
print("   Ready to interact with customers!")


 Customer Support Agent initialized successfully!
   Model in use : gemini-2.5-flash-lite
   Sub-agents   : 1 (remote catalog agent via A2A)
   Ready to interact with customers!


In [ ]:
async def test_customer_support_a2a(query: str):
    """
    Verify A2A interaction between the Customer Support Agent and the Catalog Lookup Agent.

    Steps performed:
    1. Initialize a new conversation session
    2. Dispatch the user query to the Customer Support Agent
    3. Support Agent interacts with the remote Catalog Agent via A2A
    4. Print the final agent response

    Args:
        query (str): The message to send to the Customer Support Agent
    """
    # Initialize in-memory session management
    session_mgr = InMemorySessionService()

    # Define session parameters
    application_name = "support_app"
    user_identifier = "demo_user"
    session_identifier = f"session_{uuid.uuid4().hex[:8]}"  # Unique session per test

    # IMPORTANT: Sessions must be created before running the agent
    session = await session_mgr.create_session(
        app_name=application_name,
        user_id=user_identifier,
        session_id=session_identifier
    )

    # Create a runner for managing the Customer Support Agent
    agent_runner = Runner(
        agent=support_agent,
        app_name=application_name,
        session_service=session_mgr
    )

    # Wrap user query into Content/Part objects
    message_content = types.Content(parts=[types.Part(text=query)])

    # Display the user query
    print(f"\n👤 User Query: {query}")
    print("\n🎧 Agent Response:")
    print("=" * 60)

    # Execute the agent asynchronously and stream responses
    async for event in agent_runner.run_async(
        user_id=user_identifier,
        session_id=session_identifier,
        new_message=message_content
    ):
        # Only print the final response (ignore intermediate streaming events)
        if event.is_final_response() and event.content:
            for part in event.content.parts:
                if hasattr(part, "text"):
                    print(part.text)

    print("=" * 60)



In [ ]:
# Example test execution
print("Starting A2A communication test...\n")
await test_customer_support_a2a("Can you provide details on the iPhone 15 Pro? Is it currently in stock?")

Starting A2A communication test...


👤 User Query: Can you provide details on the iPhone 15 Pro? Is it currently in stock?

🎧 Agent Response:


CancelledError: 

In [ ]:
# Test comparing multiple products
await test_customer_support_a2a(
    "I'm looking for a laptop. Can you compare the Dell XPS 15 and MacBook Pro 14 for me?"
)

In [ ]:
# Test specific product inquiry
await test_customer_support_a2a(
    "Do you have the Sony WH-1000XM5 headphones? What's the price?"
)

A2A (Agent-to-Agent) Communication Overview


A2A enables multiple AI agents to collaborate seamlessly, sharing information and tasks automatically.

One agent can act as a client/proxy to another agent, querying it as if it were local.

Workflow Implemented

Product Catalog Agent: Exposes product information and availability via an A2A-compatible server.

Customer Support Agent: Uses the remote catalog agent as a sub-agent to answer customer queries.

RemoteA2aAgent: Serves as a client-side proxy for the catalog agent, allowing the support agent to query it over HTTP using the A2A protocol.

Test Function: Demonstrates end-to-end interaction, where the support agent fetches product data from the catalog agent and returns it to the user.

Key Advantages

Separation of concerns: Each agent focuses on a specific task.

Reusability: Catalog agent can serve multiple other agents.

Scalability: Additional agents can be added without changing existing agents.

Realistic simulation of multi-agent workflows for industry use cases.

In [ ]:
explore further:

https://google.github.io/adk-docs/a2a/intro/
https://google.github.io/adk-docs/a2a/quickstart-exposing/
https://google.github.io/adk-docs/a2a/quickstart-consuming/